<a href="https://colab.research.google.com/github/DABMASTER-Brought-me-into-this/ZeroToHeroColabCollection/blob/main/makemorept4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib
import torch
import torch.nn.functional as F
import random
%matplotlib inline

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2026-04-26 17:30:33--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.006s  

2026-04-26 17:30:33 (33.8 MB/s) - ‘names.txt’ saved [228145/228145]



In [ ]:
# Retrieving all the names
data = open('names.txt').read().splitlines()
random.shuffle(data)

# Getting all unique letters
every_letter = sorted(list(set(''.join(data))))
stoi = {letter: count + 1 for count, letter in enumerate(every_letter)}
stoi['.'] = 0
itos = {val: key for key, val in stoi.items()}

In [ ]:
def build_dataset(words):
  X, Y = [], []
  for word in words:
    context = [0, 0, 0]
    for w in word + '.':
      X.append(context)
      Y.append(stoi[w])
      context = context[1:] + [stoi[w]]

  return X, Y

n = int(len(data)*0.8)
Xtr, Ytr = build_dataset(data[:n])
Xtr, Ytr = torch.tensor(Xtr), torch.tensor(Ytr)
Xte, Yte = build_dataset(data[n:]) # I aint doing 80-10-10 cuz we didnt learn how to fine tune hyper parameters
Xte, Yte = torch.tensor(Xte), torch.tensor(Yte)

In [ ]:
# Utility Function to compare manual gradient vs PyTorch Gradient
def cmp(s, dt, t):
  ex = torch.all(dt == t.grad).item()
  app = torch.allclose(dt, t.grad)
  maxdiff = (dt - t.grad).abs().max().item()
  print(f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}")

In [ ]:
# Create the coordinate tensor
C = torch.randn(27, 2)
C.requires_grad = True
emb = C[Xtr]
emb = emb.view(emb.shape[0], emb.shape[1] * emb.shape[2])

In [ ]:
# Number of Neurons
non = 100
batch_size = 32
n = batch_size

In [ ]:
# Creating Weights & Biases
# We initialize them first, then set requires_grad to True so they are 'leaf' tensors.
W1 = torch.randn((3 * 2), non) * (5/3)/(emb.shape[0] * 3)
B1 = torch.randn(non) * 0.1
W2 = torch.randn(non, 27) * 0.1
B2 = torch.randn(27) * 0.1

bngain = torch.ones((1, non)) * 0.1 + 1
bnbias = torch.zeros((1, non)) * 0.1

parameters = [W1, B1, W2, B2, C, bngain, bnbias]

# This makes them leaf nodes so .grad is populated
for p in parameters:
    p.requires_grad = True


##My miserable Attempt at recreating the forward pass as simplified as possible

In [ ]:
# ### Rebuilding Forward Pass As Simple Operations As Possible


## Random Sampling from Xtr
smpl = torch.randint(0, Xtr.shape[0], (batch_size, ))
Xb = Xtr[smpl]
Yb = Ytr[smpl]

# ## First Layer

# # Linear
# prod1 = emb @ W1 + B1

# # BatchNorm Layer
# sum1 = prod1.sum(dim=0, keepdim=True)
# mean1 = sum1/prod1.shape[0]
# dif1 = prod1 - mean1
# exp1 = dif1**2
# sum2 = exp1.sum(dim = 0, keepdim=True)
# quo1 = sum2 / (prod1.shape[0] - 1)
# std1 = (quo1 + 1e-5) ** 0.5
# bnraw = dif1 / std1
# layer1 = bngain * bnraw + bnbias
# layer1 = layer1.tanh()


# ## Second Layer

# # Linear
# prod2 = layer1 @ W2 + B2

# # Cross Entropy
# logitsa = prod2.exp()
# logitsb = logitsa.sum(dim=1, keepdim=True)
# logits = logitsa/logitsb
# logprob = logits.log()
# target_logits = logprob[torch.arange(0, batch_size), Ytr[smpl]]
# nll = target_logits.mean(dim=0, keepdim=True) * -1
# nll

# # PyTorch backward pass
# for p in parameters:
#   p.grad = None
# for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
#           norm_logits, logit_maxes, logits, h, hpreact, bnraw,
#          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
#          embcat, emb]:
#   t.retain_grad()
# loss.backward()
# loss

## Karpathy's Way of Simplifying the Forward Pass

In [ ]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + B1 # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact) # hidden layer
# Linear layer 2
logits = h @ W2 + B2 # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(batch_size), Yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss

tensor(3.3020, grad_fn=<NegBackward0>)

In [ ]:
## Backpropping Loss
# Cross Entropy Function
dloss = torch.tensor(1.0)
dlogprobs = torch.zeros(logprobs.shape)
dlogprobs[range(batch_size), Yb] += -1/batch_size * dloss
dprobs = dlogprobs * 1/probs
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim = True)
dcounts_sum = (-1 * counts_sum ** -2) * dcounts_sum_inv
dcounts = ((counts_sum_inv * torch.ones(counts.shape)) * dprobs) + (dcounts_sum * torch.ones(counts.shape))
dnorm_logits = counts * dcounts
dlogit_maxes = -1 * dnorm_logits.sum(1, keepdim=True)
dlogits = (dnorm_logits) + (dlogit_maxes * (logits == logit_maxes))
# Linear Layer 2
dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
# Non-linearity
dhpreact = dh * (1 - h**2)
# BatchNorm Layer
dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
dbnbias = dhpreact.sum(0, keepdim=True)
dbnraw = dhpreact * bngain
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = (-0.5 * (bnvar + 1e-5) ** -1.5) * dbnvar_inv
dbndiff2 = dbnvar * (1/(n-1)) * torch.ones(bndiff2.shape)
dbndiff = (dbndiff2 * (2 * bndiff)) + (dbnraw * bnvar_inv)
dbnmeani = (-1 * dbndiff).sum(0, keepdim=True)
dhprebn = (dbndiff) + (1/n * dbnmeani)
# Linear Layer 1
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demb[k, j]

cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, B2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, B1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: False | approximate: True  | maxdiff: 8.032657206058502e-09
h               | exact: False | approximate: False | maxdiff: 2.4365726858377457e-07
W2              | exact: False | approximate: True  | maxdiff: 2.2532731236424297e-10
b2              | exact: False | approximate: True  | maxdiff: 1.1175870895385742e-08
hpreact         | exact: False | approximate: False | maxdiff: 2.4365726858377457e-07


TypeError: all() received an invalid combination of arguments - got (bool), but expected one of:
 * (Tensor input, *, Tensor out = None)
 * (Tensor input, tuple of ints dim = None, bool keepdim = False, *, Tensor out = None)
 * (Tensor input, int dim, bool keepdim = False, *, Tensor out = None)
 * (Tensor input, name dim, bool keepdim = False, *, Tensor out = None)


In [ ]:
# Exercise 2: backprop through cross_entropy but all in one go
# to complete this challenge look at the mathematical expression of the loss,
# take the derivative, simplify the expression, and just write it out

# forward pass
loss_fast = -((((logits - logits.max(1,keepdim = True).values).exp() * ((logits - logits.max(1,keepdim=True).values).exp()).sum(1,keepdim=True) ** -1).log())[range(n), Yb]).mean()
# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
# loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.301966905593872 diff: 0.0


In [ ]:
# backward pass

# -----------------
# YOUR CODE HERE :)
dlogits = (((logits.clone() - logits.clone().max(1, keepdim=True).values).exp()) / ((logits.clone() - logits.clone().max(1, keepdim=True).values).exp()).sum(1, keepdim = True))
dlogits[range(n), Yb] -= 1
dlogits *= 1/n
# -----------------

cmp('logits', dlogits, logits) # I can only get approximate to be true, my maxdiff is 6e-9

logits          | exact: False | approximate: True  | maxdiff: 8.032657206058502e-09


In [ ]:
dlogits.shape, logits.shape, Yb.view(-1,1).shape

(torch.Size([32, 27]), torch.Size([32, 27]), torch.Size([32, 1]))

In [ ]:
# Exercise 3: backprop through batchnorm but all in one go
# to complete this challenge look at the mathematical expression of the output of batchnorm,
# take the derivative w.r.t. its input, simplify the expression, and just write it out
# BatchNorm paper: https://arxiv.org/abs/1502.03167

# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(9.3132e-10, grad_fn=<MaxBackward1>)


In [ ]:
# backward pass

# before we had:
# dbnraw = bngain * dhpreact
# dbndiff = bnvar_inv * dbnraw # Hola
# dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
# dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
# dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
# dbndiff += (2*bndiff) * dbndiff2 # Hola
# dhprebn = dbndiff.clone() # Important
# dbnmeani = (-dbndiff).sum(0)
# dhprebn += 1.0/n * (torch.ones_like(hprebn) * dbnmeani) # Important

# calculate dhprebn given dhpreact (i.e. backprop through the batchnorm)
# (you'll also need to use some of the variables from the forward pass up above)

# -----------------
# YOUR CODE HERE :)
dhprebn = (bnvar_inv * (bngain * dhpreact) + (2*bndiff) * ((1.0/(n-1))*torch.ones_like(bndiff2) * ((-0.5*(bnvar + 1e-5)**-1.5) * ((bndiff * bngain * dhpreact).sum(0, keepdim=True))))) + (1.0/n * (torch.ones_like(hprebn) * (-(bnvar_inv * (bngain * dhpreact) + (2*bndiff) * ((1.0/(n-1))*torch.ones_like(bndiff2) * ((-0.5*(bnvar + 1e-5)**-1.5) * ((bndiff * (bngain * dhpreact)).sum(0, keepdim=True)))))).sum(0)))
# -----------------

cmp('hprebn', dhprebn, hprebn) # I can only get approximate to be true, my maxdiff is 9e-10

hprebn          | exact: True  | approximate: True  | maxdiff: 0.0


SyntaxError: unmatched ']' (1890416573.py, line 1)

In [ ]:
# Developing the training loop
for k in range(10000):
  # Randomly Sampling from Xtr
  smpl = [random.randint(1, Xtr.shape[0]-1) for _ in range(batch_size)]
  Xb = Xtr[smpl]
  Yb = Ytr[smpl]
  emb = C[Xb]

  # First layer
  embcat = emb.view(emb.shape[0], -1)
  hprebn = embcat @ W1 + B1

  # Explicit forward pass for variables needed in manual backprop
  bnmeani = 1/n*hprebn.sum(0, keepdim=True)
  bndiff = hprebn - bnmeani
  bndiff2 = bndiff**2
  bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)
  bnvar_inv = (bnvar + 1e-5)**-0.5
  bnraw = bndiff * bnvar_inv
  hpreact = bngain * bnraw + bnbias
  h = hpreact.tanh()

  logits = h @ W2 + B2
  loss = F.cross_entropy(logits, Yb)

  if not k % 100:
    print(k, loss.item())

  # Resetting Grads
  for p in parameters:
    p.grad = None

  # Calculating the New Loss MANUALLY
  ## Backpropping Loss
  # Cross Entropy Function
  loss.grad = torch.tensor(1.0)
  # dlogprobs = torch.zeros(logprobs.shape)
  # dlogprobs[range(batch_size), Yb] += -1/batch_size * dloss
  # dprobs = dlogprobs * 1/probs
  # dcounts_sum_inv = (counts * dprobs).sum(1, keepdim = True)
  # dcounts_sum = (-1 * counts_sum ** -2) * dcounts_sum_inv
  # dcounts = ((counts_sum_inv * torch.ones(counts.shape)) * dprobs) + (dcounts_sum * torch.ones(counts.shape))
  # dnorm_logits = counts * dcounts
  # dlogit_maxes = -1 * dnorm_logits.sum(1, keepdim=True)
  # dlogits = (dnorm_logits) + (dlogit_maxes * (logits == logit_maxes))
  logits.grad = (((logits.clone() - logits.clone().max(1, keepdim=True).values).exp()) / ((logits.clone() - logits.clone().max(1, keepdim=True).values).exp()).sum(1, keepdim = True))
  logits.grad[range(batch_size), Yb] -= 1.0
  logits.grad *= 1/batch_size

  # Linear Layer 2
  # dh = dlogits @ W2.T
  # dW2 = h.T @ dlogits
  # db2 = dlogits.sum(0)
  h.grad = logits.grad @ W2.T
  W2.grad = h.T @ logits.grad
  B2.grad = logits.grad.sum(0)

  # Non-linearity
  # dhpreact = dh * (1 - h**2)
  hpreact.grad = h.grad * (1 - h**2)

  # BatchNorm Layer
  # dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
  # dbnbias = dhpreact.sum(0, keepdim=True)
  # dbnraw = dhpreact * bngain
  # dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
  # dbnvar = (-0.5 * (bnvar + 1e-5) ** -1.5) * dbnvar_inv
  # dbndiff2 = dbnvar * (1/(n-1)) * torch.ones(bndiff2.shape)
  # dbndiff = (dbndiff2 * (2 * bndiff)) + (dbnraw * bnvar_inv)
  # dbnmeani = (-1 * dbndiff).sum(0, keepdim=True)
  # dhprebn = (dbndiff) + (1/n * dbnmeani)
  bngain.grad = (hpreact.grad * bnraw).sum(0, keepdim=True)
  bnbias.grad = hpreact.grad.sum(0, keepdim=True)
  hprebn.grad = (bnvar_inv * (bngain * hpreact.grad) + (2*bndiff) * ((1.0/(n-1))*torch.ones_like(bndiff2) * ((-0.5*(bnvar + 1e-5)**-1.5) * ((bndiff * bngain * hpreact.grad).sum(0, keepdim=True))))) + (1.0/n * (torch.ones_like(hprebn) * (-(bnvar_inv * (bngain * hpreact.grad) + (2*bndiff) * ((1.0/(n-1))*torch.ones_like(bndiff2) * ((-0.5*(bnvar + 1e-5)**-1.5) * ((bndiff * (bngain * hpreact.grad)).sum(0, keepdim=True)))))).sum(0)))

  # Linear Layer 1
  # dembcat = dhprebn @ W1.T
  # dW1 = embcat.T @ dhprebn
  # db1 = dhprebn.sum(0)
  embcat.grad = hprebn.grad @ W1.T
  W1.grad = embcat.T @ hprebn.grad
  B1.grad = hprebn.grad.sum(0)

  # demb = dembcat.view(emb.shape)
  emb.grad = embcat.grad.view(emb.shape)
  C.grad = torch.zeros_like(C)
  for i in range(Xb.shape[0]):
      for j in range(Xb.shape[1]):
          ix = Xb[i, j]
          C.grad[ix] += emb.grad[i, j]

  # Updating Data
  for p in parameters:
    p.data += -0.01 * p.grad

0 4.346566200256348
100 3.356360912322998
200 3.122162342071533
300 3.0492944717407227
400 2.8688430786132812
500 3.428168773651123
600 2.7460732460021973
700 2.854602575302124
800 2.732897996902466
900 2.867727041244507
1000 2.702256202697754
1100 2.458850860595703
1200 2.941606283187866
1300 2.9927265644073486
1400 2.9366867542266846
1500 2.7057154178619385
1600 2.5616588592529297
1700 2.83412504196167
1800 2.6220004558563232
1900 2.59013295173645
2000 2.7407522201538086
2100 2.610374927520752
2200 2.716169595718384
2300 2.5947675704956055
2400 2.468836784362793
2500 2.618467092514038
2600 2.702315092086792
2700 2.5707175731658936
2800 2.409903049468994
2900 2.7392945289611816
3000 2.836683750152588
3100 2.828484058380127
3200 2.791437864303589
3300 2.612468957901001
3400 2.7867889404296875
3500 2.513235569000244
3600 2.4205515384674072
3700 2.6485705375671387
3800 2.5100903511047363
3900 2.639167547225952
4000 2.381627082824707
4100 2.5793604850769043
4200 2.63704514503479
4300 2.23

KeyboardInterrupt: 